In [1]:
# ==========================================
# ENVIRONMENT SETUP
# ==========================================
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

def find_project_root(markers=(".git", "pyproject.toml", "src")):
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if any((parent / marker).exists() for marker in markers):
            return parent
    raise RuntimeError("Project root not found.")

project_root = str(find_project_root())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
# ==========================================
# IMPORTS
# ==========================================
import pandas as pd
from experiments.scripts.EXP_013_MP_TYLER_H0_CALIBRATION import exp_013_mp_tyler_h0_calibration

In [3]:
# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    "n": 400,
    "p": 200,
    "M": 200,
    "seed": 42
}

In [4]:
# ==========================================
# RUN EXPERIMENT
# ==========================================
output = exp_013_mp_tyler_h0_calibration(**CONFIG)
results = output["results"]
meta = output["meta"]

Running Tyler H0 Duel: 100%|██████████| 200/200 [00:14<00:00, 13.35it/s]


In [5]:
# ==========================================
# METADATA
# ==========================================
print("\n=== METADATA ===")
for k, v in meta.items():
    print(f"{k}: {v}")


=== METADATA ===
n: 400
p: 200
q: 0.5
M: 200
seed: 42
execution_time_minutes: 0.26
model_version: 0.1.0


In [7]:
# ==========================================
# RESULTS AND TABLE
# ==========================================
print("=" * 55)
print(" EXPERIMENT 013: TYLER VS PEARSON (H0) ")
print("=" * 55)

print("\n=== INTERNAL TELEMETRY (THE WHY) ===")
tel = results['telemetry']
print(f"Variance (sigma2_hat) : Pearson={tel['sigma2_pearson']:.4f} | Tyler={tel['sigma2_tyler']:.4f}")
print(f"Analytic Boundary (λ+): Pearson={tel['lambda_plus_pearson']:.4f} | Tyler={tel['lambda_plus_tyler']:.4f}")

print("\n=== CALIBRATION RESULTS (THE WHAT) ===")
print(f"FPR Pearson (Control) : {results['fpr_pearson']:.4f}")
print(f"FPR Tyler (Pathology) : {results['fpr_tyler']:.4f}  <-- CATASTROPHIC FAILURE")

print("\nTyler k_effective Distribution:")
df_dist = pd.DataFrame.from_dict(results["distribution_tyler"], orient='index', columns=['Count'])
display(df_dist)

 EXPERIMENT 013: TYLER VS PEARSON (H0) 

=== INTERNAL TELEMETRY (THE WHY) ===
Variance (sigma2_hat) : Pearson=0.9992 | Tyler=0.9946
Analytic Boundary (λ+): Pearson=2.9120 | Tyler=2.8985

=== CALIBRATION RESULTS (THE WHAT) ===
FPR Pearson (Control) : 0.0800
FPR Tyler (Pathology) : 0.4950  <-- CATASTROPHIC FAILURE

Tyler k_effective Distribution:


,Count
0,101
1,88
2,10
3,1


### Interpretation

1. **Theoretical Alignment (The Breakdown of Independence):** The Marchenko-Pastur theorem relies absolutely on the assumption of independent matrix entries. The data demonstrates that Tyler's robust M-estimator violently breaks this assumption. While the classical Pearson estimator remains highly calibrated (FPR $\approx 8.0\%$, reflecting only standard finite-sample Tracy-Widom leakage), Tyler's estimator suffers a catastrophic failure, hallucinating spurious factors half of the time (FPR $\approx 49.5\%$). 

2. **Finite-Sample Mechanics (Trace Constraint Deformation):** The internal telemetry reveals the mechanical cause of this failure. Tyler's estimator projects the spatial data onto the unit sphere and enforces a strict trace constraint ($\text{tr}(\Sigma)=p$). This normalization introduces repulsive angular dependencies between the samples, fundamentally altering the topology of the spectral bulk. The mathematically calculated analytic boundary ($\lambda_+ \approx 2.89$) is structurally too narrow to contain these artificially fattened tails, causing 1 to 3 noise eigenvalues to consistently spill over the limit.

3. **Pipeline Justification (The Bootstrap Mandate for Robustness):** This result formalizes a critical architectural rule: you cannot safely mix robust spatial estimators with classical analytical thresholds. If the production environment requires Tyler's covariance to survive extreme outliers or heavy tails, it mathematically disqualifies the Marchenko-Pastur limit. Therefore, whenever `covariance="tyler"` is invoked in the API, the `threshold="bootstrap"` module becomes strictly mandatory to map the deformed, non-standard null distribution and restore absolute Type I error control.